# Lipid RT — RDKit-only baseline + ChemBERTa-Concat ablation

Two new model conditions on the METLIN lipid RT split, evaluated alongside the paper's existing single-task / multi-task models:

1. **RDKit-only baseline** — Ridge, RandomForest, GradientBoosting on the seven RDKit descriptors (no ChemBERTa). Establishes the floor: how much of the lipid RT signal is already in the rule-based descriptors?
2. **ChemBERTa-Concat** — ChemBERTa CLS embedding concatenated with the seven RDKit descriptors, fed through a single linear head. Single-task RT loss only. Tests whether multi-task auxiliary supervision contributes beyond what the descriptors themselves already provide.

Same train/test split, RT scaling convention, optimizer, batch size, and epoch count as `chemberta_lipid_rt_RTrdkit.ipynb`. Output: long-format CSVs (`model, split, epoch, metric, value`) compatible with the paper's existing supplementary data tables.

In [ ]:
!pip install torch transformers scikit-learn matplotlib joblib

In [ ]:
# Upload METLIN_RT_lipid_train_with_rdkit.csv and METLIN_RT_lipid_test_with_rdkit.csv
n = 1
while n < 2:
    from google.colab import files
    uploaded = files.upload()
    n += 1

In [ ]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('/content/METLIN_RT_lipid_train_with_rdkit.csv')
test_df  = pd.read_csv('/content/METLIN_RT_lipid_test_with_rdkit.csv')
print('train:', len(train_df), '  test:', len(test_df))

DESC_COLS = ['mol_weight', 'polar_surface_area', 'h_bond_donors',
             'h_bond_acceptors', 'rotatable_bonds', 'aromatic_rings', 'heavy_atoms']

## 1. RDKit-only baseline

No ChemBERTa, no SMILES tokens — just the seven descriptors → RT. If this gets close to the ChemBERTa numbers, the entire LLM pipeline is doing very little work.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error

X_train = train_df[DESC_COLS].values.astype(float)
X_test  = test_df[DESC_COLS].values.astype(float)
y_train = train_df['rt'].values.astype(float)
y_test  = test_df['rt'].values.astype(float)

x_scaler = MinMaxScaler().fit(X_train)
X_train_s = x_scaler.transform(X_train)
X_test_s  = x_scaler.transform(X_test)

# Match the RT-scaling convention of the existing notebooks (fit MinMax on train RT only).
y_scaler = MinMaxScaler()
y_train_s = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test_s  = y_scaler.transform(y_test.reshape(-1, 1)).ravel()

BASELINE_SEEDS = [0, 1, 2, 3, 4]
rows = []

for seed in BASELINE_SEEDS:
    candidates = [
        ('Ridge',            Ridge(alpha=1.0)),
        ('RandomForest',     RandomForestRegressor(n_estimators=200, random_state=seed, n_jobs=-1)),
        ('GradientBoosting', GradientBoostingRegressor(n_estimators=200, random_state=seed)),
    ]
    for name, mdl in candidates:
        mdl.fit(X_train_s, y_train_s)
        for split_name, X_s, y_orig in [('Train', X_train_s, y_train), ('Test', X_test_s, y_test)]:
            y_pred_s = mdl.predict(X_s)
            y_pred   = y_scaler.inverse_transform(y_pred_s.reshape(-1, 1)).ravel()
            rows.append({'model': f'RDKit-only ({name})', 'seed': seed, 'split': split_name,
                         'metric': 'R2',  'value': r2_score(y_orig, y_pred)})
            rows.append({'model': f'RDKit-only ({name})', 'seed': seed, 'split': split_name,
                         'metric': 'MAE', 'value': mean_absolute_error(y_orig, y_pred)})

baseline_df = pd.DataFrame(rows)
baseline_df.to_csv('lipid_rdkit_only_baseline_metrics.csv', index=False)
print(baseline_df.pivot_table(index='model', columns=['split', 'metric'], values='value',
                              aggfunc=['median', 'std']).round(4))

## 2. ChemBERTa-Concat

[CLS] embedding (768) ‖ scaled RDKit descriptors (7) → Linear → RT. Single-task MSE on RT. Same hyperparameters as the paper's other ChemBERTa runs (AdamW lr=2.5e-5, batch=16, 15 epochs).

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# Re-fit a descriptor scaler dedicated to the concat model's inputs (kept separate from the
# baseline's x_scaler purely to avoid hidden coupling between sections).
desc_scaler = MinMaxScaler().fit(train_df[DESC_COLS].values)
desc_train  = desc_scaler.transform(train_df[DESC_COLS].values).astype(np.float32)
desc_test   = desc_scaler.transform(test_df[DESC_COLS].values).astype(np.float32)

rt_scaler = MinMaxScaler()
rt_train  = rt_scaler.fit_transform(train_df[['rt']].values).astype(np.float32).ravel()
rt_test   = rt_scaler.transform(test_df[['rt']].values).astype(np.float32).ravel()

class LipidConcatDataset(Dataset):
    def __init__(self, smiles, descriptors, rt, max_length=128):
        self.smiles = smiles
        self.descriptors = torch.tensor(descriptors, dtype=torch.float32)
        self.rt = torch.tensor(rt, dtype=torch.float32)
        self.max_length = max_length
    def __len__(self):
        return len(self.smiles)
    def __getitem__(self, idx):
        enc = tokenizer(self.smiles[idx], padding='max_length', truncation=True,
                        max_length=self.max_length, return_tensors='pt')
        return {
            'input_ids':       enc['input_ids'].squeeze(0),
            'attention_mask':  enc['attention_mask'].squeeze(0),
            'descriptors':     self.descriptors[idx],
            'rt':              self.rt[idx],
        }

train_ds = LipidConcatDataset(train_df['smile'].tolist(), desc_train, rt_train)
test_ds  = LipidConcatDataset(test_df['smile'].tolist(),  desc_test,  rt_test)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False)

In [ ]:
class ChemBERTaConcatRegressor(nn.Module):
    def __init__(self, n_descriptors=7):
        super().__init__()
        self.bert = AutoModel.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')
        hs = self.bert.config.hidden_size
        self.regressor = nn.Linear(hs + n_descriptors, 1)
    def forward(self, input_ids, attention_mask, descriptors):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        x = torch.cat([cls, descriptors], dim=-1)
        return self.regressor(x)

def evaluate(model, loader):
    model.eval()
    preds, ys = [], []
    with torch.no_grad():
        for batch in loader:
            p = model(batch['input_ids'].to(device),
                      batch['attention_mask'].to(device),
                      batch['descriptors'].to(device)).squeeze(-1).cpu().numpy()
            preds.append(p); ys.append(batch['rt'].numpy())
    p = rt_scaler.inverse_transform(np.concatenate(preds).reshape(-1, 1)).ravel()
    y = rt_scaler.inverse_transform(np.concatenate(ys).reshape(-1, 1)).ravel()
    return r2_score(y, p), mean_absolute_error(y, p)

# Set CONCAT_SEEDS = [0, 1, 2, 3, 4] for the multi-seed run that addresses Reviewer 1's
# variance complaint. Keep it [0] for an initial dev/sanity pass.
CONCAT_SEEDS = [0]
EPOCHS = 15

concat_rows = []

for seed in CONCAT_SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = ChemBERTaConcatRegressor(n_descriptors=len(DESC_COLS)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2.5e-5)
    criterion = nn.MSELoss()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            pred = model(batch['input_ids'].to(device),
                         batch['attention_mask'].to(device),
                         batch['descriptors'].to(device)).squeeze(-1)
            loss = criterion(pred, batch['rt'].to(device))
            loss.backward()
            optimizer.step()

        r2_tr, mae_tr = evaluate(model, train_loader)
        r2_te, mae_te = evaluate(model, test_loader)
        for split_name, r2, mae in [('Train', r2_tr, mae_tr), ('Test', r2_te, mae_te)]:
            concat_rows.append({'model': 'ChemBERTa-Concat', 'seed': seed, 'epoch': epoch,
                                'split': split_name, 'metric': 'R2',  'value': r2})
            concat_rows.append({'model': 'ChemBERTa-Concat', 'seed': seed, 'epoch': epoch,
                                'split': split_name, 'metric': 'MAE', 'value': mae})
        print(f'seed={seed}  epoch={epoch:2d}  R2_test={r2_te:.4f}  MAE_test={mae_te:.2f}')

concat_df = pd.DataFrame(concat_rows)
concat_df.to_csv('lipid_chemberta_concat_metrics.csv', index=False)

## 3. Quick comparison

Summary table for the response-to-reviewers. Pull the existing single-task and multi-task numbers from the paper's Supplementary Data 2 (`Lipid_ChemBERTa_RDkit_Boxplot_Data_Long.csv`) and append them here once you have it; the column schema (`model, split, epoch, metric, value`) matches by design.

In [ ]:
def summarise(df, group_cols=('model', 'split', 'metric')):
    g = df.groupby(list(group_cols))['value']
    return pd.DataFrame({
        'median': g.median(),
        'iqr_low':  g.quantile(0.25),
        'iqr_high': g.quantile(0.75),
        'n':        g.size(),
    }).round(4)

print('=== RDKit-only baseline (across seeds) ===')
print(summarise(baseline_df))

print('\n=== ChemBERTa-Concat (across seeds × epochs) ===')
print(summarise(concat_df))